In [1]:
import pandas as pd
from torch_geometric.data import DataLoader
from tqdm import tqdm
# Download and process data at './dataset/ogbg_molhiv/'
from ogb.nodeproppred import PygNodePropPredDataset, Evaluator
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
import numpy as np
import seaborn as sns
import torch_geometric.transforms as T
import networkx as nx
import scipy.sparse as sp
# 处理成Dataframe()格式是因为在Dataframe()格式下处理数据方便一些

Using backend: pytorch


In [35]:
d = 'ogbn-mag'
dataset = PygNodePropPredDataset(name = d) 

In [3]:
graph = dataset[0]

In [4]:
graph

Data(
  num_nodes_dict={
    author=1134649,
    field_of_study=59965,
    institution=8740,
    paper=736389
  },
  edge_index_dict={
    (author, affiliated_with, institution)=[2, 1043998],
    (author, writes, paper)=[2, 7145660],
    (paper, cites, paper)=[2, 5416271],
    (paper, has_topic, field_of_study)=[2, 7505078]
  },
  x_dict={ paper=[736389, 128] },
  node_year={ paper=[736389, 1] },
  edge_reltype={
    (author, affiliated_with, institution)=[1043998, 1],
    (author, writes, paper)=[7145660, 1],
    (paper, cites, paper)=[5416271, 1],
    (paper, has_topic, field_of_study)=[7505078, 1]
  },
  y_dict={ paper=[736389, 1] }
)

In [30]:
def get_adj(edge_index):
    N = edge_index.shape[1]
    data = np.ones(N)
    row_ind = edge_index[0]
    col_ind = edge_index[1]
    adj = sp.csr_matrix((data, (row_ind, col_ind)), shape=(edge_index.max() + 1, edge_index.max() + 1))
    return adj

# mag

In [22]:
labels = graph.y_dict['paper'].numpy().ravel()

In [33]:
labels

array([246, 131, 189, ..., 266, 289,   1], dtype=int64)

In [26]:
features = graph.x_dict['paper'].numpy()

In [34]:
features

array([[-0.095379,  0.040758, -0.210948, ...,  0.061569, -0.027663,
        -0.133832],
       [-0.151047, -0.107315, -0.221964, ...,  0.345754, -0.027737,
        -0.218527],
       [-0.114799, -0.175982, -0.260556, ...,  0.173058, -0.156445,
        -0.277954],
       ...,
       [ 0.022815, -0.0865  ,  0.098138, ..., -0.054667, -0.207721,
        -0.230458],
       [-0.289148, -0.202898, -0.152454, ...,  0.104207,  0.204123,
        -0.352805],
       [-0.088966, -0.034788, -0.264226, ...,  0.260077, -0.087453,
        -0.517127]], dtype=float32)

In [31]:
edge_index = graph.edge_index_dict[('paper', 'cites','paper')]
adj = get_adj(edge_index)

In [32]:
adj

<736389x736389 sparse matrix of type '<class 'numpy.float64'>'
	with 5416271 stored elements in Compressed Sparse Row format>

In [36]:
np.savez('C://Users/pc/Desktop/' + d + '.npz', adj_matrix=adj, node_attr=features, node_label=labels)

# 异质

In [16]:
from tao_utils import *
def load_h(fp, dataset_name):
    graph_adjacency_list_file_path = os.path.join(fp, dataset_name, 'out1_graph_edges.txt')
    graph_node_features_and_labels_file_path = os.path.join(fp, dataset_name,
                                                            f'out1_node_feature_label.txt')

    G = nx.DiGraph()
    graph_node_features_dict = {}
    graph_labels_dict = {}

    if dataset_name == 'film':
        with open(graph_node_features_and_labels_file_path) as graph_node_features_and_labels_file:
            graph_node_features_and_labels_file.readline()
            for line in graph_node_features_and_labels_file:
                line = line.rstrip().split('\t')
                assert (len(line) == 3)
                assert (int(line[0]) not in graph_node_features_dict and int(line[0]) not in graph_labels_dict)
                feature_blank = np.zeros(932, dtype=np.uint8)
                feature_blank[np.array(line[1].split(','), dtype=np.uint16)] = 1
                graph_node_features_dict[int(line[0])] = feature_blank
                graph_labels_dict[int(line[0])] = int(line[2])
    else:
        with open(graph_node_features_and_labels_file_path) as graph_node_features_and_labels_file:
            graph_node_features_and_labels_file.readline()
            for line in graph_node_features_and_labels_file:
                line = line.rstrip().split('\t')
                assert (len(line) == 3)
                assert (int(line[0]) not in graph_node_features_dict and int(line[0]) not in graph_labels_dict)
                graph_node_features_dict[int(line[0])] = np.array(line[1].split(','), dtype=np.uint8)
                graph_labels_dict[int(line[0])] = int(line[2])

    with open(graph_adjacency_list_file_path) as graph_adjacency_list_file:
        graph_adjacency_list_file.readline()
        for line in graph_adjacency_list_file:
            line = line.rstrip().split('\t')
            assert (len(line) == 2)
            if int(line[0]) not in G:
                G.add_node(int(line[0]), features=graph_node_features_dict[int(line[0])],
                           label=graph_labels_dict[int(line[0])])
            if int(line[1]) not in G:
                G.add_node(int(line[1]), features=graph_node_features_dict[int(line[1])],
                           label=graph_labels_dict[int(line[1])])
            G.add_edge(int(line[0]), int(line[1]))

    adj = nx.adjacency_matrix(G, sorted(G.nodes()))
    features = np.array(
        [features for _, features in sorted(G.nodes(data='features'), key=lambda x: x[0])])
    labels = np.array(
        [label for _, label in sorted(G.nodes(data='label'), key=lambda x: x[0])])
    return adj, features, labels

In [17]:
dataset = 'film'
fp = 'C://Users/pc/Desktop/dataset/'
adj, features, labels = load_h(fp, dataset)

In [18]:
adj

<7600x7600 sparse matrix of type '<class 'numpy.intc'>'
	with 30019 stored elements in Compressed Sparse Row format>

In [22]:
features

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 1],
       [0, 0, 0, ..., 0, 0, 1],
       [0, 0, 0, ..., 0, 0, 0]], dtype=uint8)

In [20]:
labels

array([3, 3, 3, ..., 2, 1, 3])

In [23]:
np.savez('C://Users/pc/Desktop/'+dataset+'.npz', adj_matrix=adj, node_attr=features, node_label=labels)

# reddit

In [7]:
import json
from networkx.readwrite import json_graph
import scipy.sparse as sp
import numpy as np

In [2]:
def transferRedditDataFormat(dataset_dir='C://Users/pc/Downloads/reddit', output_file='reddit.npz'):
    G = json_graph.node_link_graph(json.load(open(dataset_dir + "/reddit-G.json")))
    labels = json.load(open(dataset_dir + "/reddit-class_map.json"))

    train_ids = [n for n in G.nodes() if not G.nodes[n]['val'] and not G.nodes[n]['test']]
    test_ids = [n for n in G.nodes() if G.nodes[n]['test']]
    val_ids = [n for n in G.nodes() if G.nodes[n]['val']]
    train_labels = [labels[i] for i in train_ids]
    test_labels = [labels[i] for i in test_ids]
    val_labels = [labels[i] for i in val_ids]
    feats = np.load(dataset_dir + "/reddit-feats.npy")
    ## Logistic gets thrown off by big counts, so log transform num comments and score
    feats[:, 0] = np.log(feats[:, 0] + 1.0)
    feats[:, 1] = np.log(feats[:, 1] - min(np.min(feats[:, 1]), -1))
    feat_id_map = json.load(open(dataset_dir + "reddit-id_map.json"))
    feat_id_map = {id: val for id, val in feat_id_map.iteritems()}

    # train_feats = feats[[feat_id_map[id] for id in train_ids]]
    # test_feats = feats[[feat_id_map[id] for id in test_ids]]

    numNode = len(feat_id_map)
    adj = sp.lil_matrix(np.zeros((numNode,numNode)))
    for edge in G.edges():
        adj[feat_id_map[edge[0]], feat_id_map[edge[1]]] = 1
    np.savez('reddit_adj.npz')

    train_index = [feat_id_map[id] for id in train_ids]
    val_index = [feat_id_map[id] for id in val_ids]
    test_index = [feat_id_map[id] for id in test_ids]
    np.savez(output_file, feats = feats, y_train=train_labels, y_val=val_labels, y_test = test_labels, train_index = train_index,
             val_index=val_index, test_index = test_index)

In [5]:
def loadRedditFromNPZ(dataset_dir):
    adj = sp.load_npz(dataset_dir+"reddit_adj.npz")
    data = np.load(dataset_dir+"reddit_.npz")
    return adj + adj.T, data['feats'], data['y_train'], data['y_val'], data['y_test'], data['train_index'], data['val_index'], data['test_index']

In [6]:
data = np.load(dataset_dir+"reddit_.npz")

NameError: name 'np' is not defined

# fastgcn npz读取

In [8]:
dataset_dir = 'D://inpluslab/'
dataset_dir = 'C://Users/pc/Downloads/'
f_adj, f_features, y_train, y_val, y_test,train_index, val_index, test_index = loadRedditFromNPZ(dataset_dir)

In [9]:
f_adj

<232965x232965 sparse matrix of type '<class 'numpy.float64'>'
	with 23213838 stored elements in Compressed Sparse Row format>

In [34]:
f_features

array([[ 5.54126355e+00,  8.03722003e+00,  3.88128497e-03, ...,
         5.78349151e-03,  1.09679537e-02, -2.49183578e-03],
       [ 1.79175947e+00,  6.93147181e-01,  2.49943547e-02, ...,
         9.06685212e-03,  8.92978317e-03, -4.29881787e-03],
       [ 1.94591015e+00,  1.38629436e+00,  2.18348876e-02, ...,
         8.11181508e-03,  1.07228216e-02,  1.77117347e-03],
       ...,
       [ 2.99573227e+00,  6.93147181e-01,  4.19115089e-02, ...,
         1.65987819e-02, -6.24787849e-03, -1.59278626e-03],
       [ 6.93147181e-01,  6.93147181e-01,  2.06362084e-02, ...,
         1.37258768e-02,  2.91283568e-03, -1.46052791e-02],
       [ 3.87120101e+00,  5.09986643e+00,  2.59802155e-02, ...,
         9.56884861e-03,  1.37876200e-02,  3.63189922e-03]])

In [18]:
feat_id_map.items()

dict_items([('2hgylt', 0), ('2g1zkx', 1), ('2hgyls', 2), ('2f6ifj', 3), ('2h74lt', 5), ('2g9w2s', 6), ('2h5erw', 7), ('2fj5f2', 8), ('2f6ifq', 9), ('2g1zkh', 10), ('2fj5f6', 11), ('2gfwu7', 12), ('2g9w2j', 71), ('2hk2a8', 14), ('2h5erc', 15), ('2fmexp', 16), ('2fmexr', 17), ('2hgylk', 18), ('2glkrv', 19), ('2grcu8', 20), ('2glkru', 21), ('2h50et', 22), ('2f5dwb', 23), ('2hp9br', 75850), ('2fhtmq', 37639), ('2htxgr', 26), ('2h7sdb', 27), ('2glkry', 28), ('2gj940', 29), ('2f5dwu', 30), ('2g92gr', 209280), ('2g8iz4', 31), ('2h7sdu', 32), ('2glkri', 33), ('2fj5fa', 34), ('2htxg6', 35), ('2h74l0', 36), ('2f785z', 37), ('2g1zk7', 38), ('2hgyl9', 39), ('2g9w23', 40), ('2h5er7', 41), ('2f6if3', 42), ('2f785a', 43), ('2f6if1', 44), ('2huair', 45), ('2gsfk8', 46), ('2hk2av', 47), ('2g2ysi', 48), ('2fkofk', 49), ('2fkofh', 50), ('2fmexc', 51), ('2gcju7', 52), ('2h1cfx', 53), ('2hvdnh', 54), ('2fanwl', 55), ('2gj94m', 56), ('2grcuv', 57), ('2gfwu2', 58), ('2hvdnx', 59), ('2f5dw1', 60), ('2g8izv', 

In [39]:
f_features.shape

(232965, 602)

In [107]:
train_index.shape[0] + val_index.shape[0] + test_index.shape[0]

231443

In [125]:
labels = np.array(list(y_train) + list(y_val) + list(y_test))

In [111]:
nodes = list(train_index) + list(val_index) + list(test_index)

In [124]:
ff_adj = f_adj[nodes][:,nodes]

In [126]:
ff_features = f_features[nodes]

In [127]:
np.savez('C://Users/pc/Desktop/reddit2.npz', adj_matrix=ff_adj, node_attr=ff_features, node_label=labels)

In [119]:
len(nodes)

231443

In [117]:
np.max(nodes)

232964

In [118]:
np.min(nodes)

0

# 原始reddit

In [3]:
dataset_dir = 'C://Users/pc/Downloads/'

In [4]:
G = json_graph.node_link_graph(json.load(open(dataset_dir + "reddit/reddit-G.json")))

In [5]:
o_features = np.load(dataset_dir + "reddit/reddit-feats.npy")

In [6]:
feat_id_map = json.load(open(dataset_dir + "reddit/reddit-id_map.json"))

In [10]:
feat_id_map_r = {val: id for id, val in zip(feat_id_map.keys(), feat_id_map.values())}

In [8]:
o_labels = json.load(open(dataset_dir + "reddit/reddit-class_map.json"))

In [11]:
oo_labels = np.array([o_labels[feat_id_map_r[i]] for i in range(232965)])

In [12]:
nodes = feat_id_map.keys()

In [13]:
o_adj = sp.lil_matrix((len(nodes),len(nodes)))

In [14]:
for edge in G.edges():
    o_adj[edge[0], edge[1]] = 1

In [15]:
o_adj

<232965x232965 sparse matrix of type '<class 'numpy.float64'>'
	with 11606919 stored elements in List of Lists format>

In [16]:
np.savez('C://Users/pc/Desktop/reddit3.npz', adj_matrix=o_adj, node_attr=o_features, node_label=oo_labels)

In [19]:
o_features

array([[ 2.54000000e+02,  3.09300000e+03,  3.88128497e-03, ...,
         5.78349151e-03,  1.09679537e-02, -2.49183578e-03],
       [ 5.00000000e+00,  1.00000000e+00,  2.49943547e-02, ...,
         9.06685212e-03,  8.92978317e-03, -4.29881787e-03],
       [ 6.00000000e+00,  3.00000000e+00,  2.18348876e-02, ...,
         8.11181508e-03,  1.07228216e-02,  1.77117347e-03],
       ...,
       [ 1.90000000e+01,  1.00000000e+00,  4.19115089e-02, ...,
         1.65987819e-02, -6.24787849e-03, -1.59278626e-03],
       [ 1.00000000e+00,  1.00000000e+00,  2.06362084e-02, ...,
         1.37258768e-02,  2.91283568e-03, -1.46052791e-02],
       [ 4.70000000e+01,  1.63000000e+02,  2.59802155e-02, ...,
         9.56884861e-03,  1.37876200e-02,  3.63189922e-03]])

In [41]:
train_ids = []
test_ids = []
val_ids = []
nan_ids = []
for n in G.nodes():
    _v = G.nodes[n].get('val', None)
    _t = G.nodes[n].get('test', None)
    if _v is None or _t is None:
        nan_ids.append(n)
        continue
    if not G.nodes[n]['val'] and not G.nodes[n]['test']:
        train_ids.append(n)
    if G.nodes[n]['test']:
        test_ids.append(n)
    if G.nodes[n]['val']:
        val_ids.append(n)

In [42]:
print(len(train_ids), len(test_ids), len(val_ids), len(train_ids) + len(test_ids) + len(val_ids))
print(len(nan_ids))

152410 55334 23699 231443
231443


In [34]:
train_labels = [labels[i] for i in train_ids]
test_labels = [labels[i] for i in test_ids]
val_labels = [labels[i] for i in val_ids]

In [38]:
feats[:, 0] = np.log(feats[:, 0] + 1.0)
feats[:, 1] = np.log(feats[:, 1] - min(np.min(feats[:, 1]), -1))

# jiaying数据集

In [4]:
import os
import pickle
import networkx as nx
from copy import deepcopy as dcopy
import numpy as np

In [5]:
def load_pickle(fileName):
    with open(fileName, 'rb') as f:
        return pickle.load(f)
def dump_pickle(data, fileName):
    with open(fileName, 'ab') as f:
        pickle.dump(data, f)
RAWDATA_PATH = 'D://taodata/jiaying/dataset/rawdata/'
PUBLICDATA_PATH = 'D://taodata/jiaying/dataset/publicdata/'

SAMPLE_GSIZE = 150000

SAMPLE_MULGS_PATH = os.path.join(PUBLICDATA_PATH, 'graph_%d/SP_MulGs.pkl' % SAMPLE_GSIZE)
FEATURES_PATH = os.path.join(PUBLICDATA_PATH, 'graph_%d/features.dat' % SAMPLE_GSIZE)

"""读取全局的抽样有向图、特征train_x和train_y"""

df = load_pickle(FEATURES_PATH)
sp_mulG  = load_pickle(SAMPLE_MULGS_PATH)
y_cols_name = ['label']
x_cols_name = [x for x in df.columns if x not in y_cols_name]
train_x = dcopy(df[x_cols_name])
train_y = dcopy(df[y_cols_name])
pos_cnt, neg_cnt = int(train_y.sum()), int(len(train_y) - train_y.sum())
scipy_adj_matrix = nx.convert_matrix.to_scipy_sparse_matrix(sp_mulG, format='coo')

# del sp_muldG, sp_mulG, df
# gc.collect()

print('pos node cnts:', pos_cnt)
print('neg node cnts:', neg_cnt, 'pos/all ratio:', pos_cnt / (pos_cnt +neg_cnt))

pos node cnts: 494
neg node cnts: 299506 pos/all ratio: 0.0016466666666666667


In [6]:
adj = scipy_adj_matrix.tocsr()

In [7]:
features = train_x.values

In [8]:
labels = train_y.values.ravel()

In [9]:
np.savez('C://Users/pc/GraphData/datasets/bc'+str(int(SAMPLE_GSIZE/10000))+'.npz', adj_matrix=adj, node_attr=features, node_label=labels)